# 01 - Data Ingestion and Cleaning

**Master's Thesis**: Assessing AI Maturity with Advanced Methods  
**University**: TU Munich -- Management & Technology  

This notebook ingests company-level data from five authoritative sources and
produces one clean CSV per source, followed by a merged master table that uses
HG Insights AI-1000 as the base and enriches it with the IMD AI Maturity score.

| # | Source | Method | Expected N |
|---|--------|--------|------------|
| 1 | Fortune 1000 US | JSON API | ~1 000 |
| 2 | Fortune Global 500 | JSON API | ~500 |
| 3 | IMD AI Maturity Index 2025 | REST API | ~300 |
| 4 | HG Insights AI-1000 | PDF extraction | 1 000 |
| 5 | Drucker Institute 2024 | HTML table | ~250 |

---
## 0 -- Imports and Configuration

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pdfplumber

warnings.filterwarnings("ignore")

# -- Project paths (relative to the notebooks/ directory) --------------------
ROOT      = Path("..").resolve()
DATA_RAW  = ROOT / "data_raw"
DATA_CLEAN = ROOT / "data_clean"
DATA_RAW.mkdir(exist_ok=True)
DATA_CLEAN.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"
    )
}

print(f"Project root : {ROOT}")
print(f"Raw data dir : {DATA_RAW}")
print(f"Clean data dir: {DATA_CLEAN}")

Project root : D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code
Raw data dir : D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_raw
Clean data dir: D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_clean


---
## 1 -- Fortune 1000 US  (JSON API)

The website us500.com renders its tables via JavaScript, but the underlying
data is fetched from a plain JSON endpoint. We use that directly.

In [ ]:
def extract_fortune1000():
    """Fetch the Fortune 1000 US list from the us500.com JSON endpoint."""
    url = "https://us500.com/data/us1000Min-2.json"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()

    records = payload if isinstance(payload, list) else payload.get("data", payload)

    rows = []
    for r in records:
        rows.append({
            "rank":      int(r.get("rank", 0)),
            "company":   r.get("company", "").strip(),
            "ticker":    r.get("ticker", ""),
            "industry":  r.get("industry", ""),
            "city":      r.get("city", ""),
            "state":     r.get("state", ""),
            "country":   "United States",
            "revenue":   r.get("revenue", ""),
            "profit":    r.get("profit", ""),
            "assets":    r.get("assets", ""),
            "employees": r.get("employees", ""),
        })

    df = pd.DataFrame(rows)
    return df


df_fortune1000 = extract_fortune1000()

out_path = DATA_RAW / "fortune_1000_us.csv"
df_fortune1000.to_csv(out_path, index=False)

print(f"Fortune 1000 US: {len(df_fortune1000)} companies")
print(f"Saved to {out_path}")
df_fortune1000.head()

---
## 2 -- Fortune Global 500  (JSON API)

Same pattern as Fortune 1000: a hidden JSON endpoint backs the JS-rendered table.

In [ ]:
def extract_fortune_global500():
    """Fetch the Fortune Global 500 list from the us500.com JSON endpoint."""
    url = "https://us500.com/data/globalMin-2.json"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()

    records = payload if isinstance(payload, list) else payload.get("data", payload)

    rows = []
    for r in records:
        rows.append({
            "rank":      int(r.get("rank", 0)),
            "company":   r.get("company", "").strip(),
            "ticker":    r.get("ticker", ""),
            "industry":  r.get("industry", ""),
            "city":      r.get("city", ""),
            "state":     r.get("state", ""),
            "country":   r.get("country", ""),
            "revenue":   r.get("revenue", ""),
            "profit":    r.get("profit", ""),
            "assets":    r.get("assets", ""),
            "employees": r.get("employees", ""),
        })

    df = pd.DataFrame(rows)
    return df


df_global500 = extract_fortune_global500()

out_path = DATA_RAW / "fortune_global_500.csv"
df_global500.to_csv(out_path, index=False)

print(f"Fortune Global 500: {len(df_global500)} companies")
print(f"Saved to {out_path}")
df_global500.head()

---
## 3 -- IMD AI Maturity Index 2025  (REST API)

IMD exposes a CosmosDB-backed REST endpoint that returns the full ranking
for both 2024 (N=200) and 2025 (N=300). We retrieve the complete payload
and split by year.

In [ ]:
def extract_imd_ai_maturity():
    """Fetch the IMD AI Maturity Index from the public REST API."""
    url = "https://www.imd.org/wp-json/api/cosmosdb/ai-maturity"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()

    all_records = payload.get("response", payload)

    rows = []
    for r in all_records:
        rows.append({
            "year":            int(r.get("year", 0)),
            "rank":            int(r.get("index_rank", 0)),
            "company_name":    r.get("company_name", "").strip(),
            "country":         r.get("country", ""),
            "index_score":     float(r.get("index_score", 0)),
            "industry_group":  r.get("industry_group", ""),
            "forbes_rank":     int(r.get("forbes_rank", 0)),
            "company_sales":   r.get("company_sales", ""),
            "company_profits": r.get("company_profits", ""),
            "company_assets":  r.get("company_assets", ""),
        })

    df = pd.DataFrame(rows)
    return df


df_imd_all = extract_imd_ai_maturity()

df_imd_2025 = df_imd_all[df_imd_all["year"] == 2025].copy().reset_index(drop=True)
df_imd_2024 = df_imd_all[df_imd_all["year"] == 2024].copy().reset_index(drop=True)

# Save all three variants
df_imd_all.to_csv(DATA_RAW / "imd_ai_maturity_all.csv", index=False)
df_imd_2025.to_csv(DATA_RAW / "imd_ai_maturity_2025.csv", index=False)
df_imd_2024.to_csv(DATA_RAW / "imd_ai_maturity_2024.csv", index=False)

print(f"IMD AI Maturity (all years): {len(df_imd_all)} records")
print(f"  2025 ranking: {len(df_imd_2025)} companies")
print(f"  2024 ranking: {len(df_imd_2024)} companies")
print(f"Saved to {DATA_RAW}")
df_imd_2025.head(10)

---
## 4 -- HG Insights AI-1000  (PDF extraction)

The full 1000-company list lives inside a 59-page PDF report. Data tables span
pages 6 through 55 (1-indexed), with 20 rows per page.  Two edge cases require
special handling:

- **Rank 1,000** is stored as `"1,000"` (with a thousands separator), so a
  naive `isdigit()` check would miss it.
- **Ranks 406/407** have a cell-merge artefact in the PDF where company names
  bleed across rows.  We detect and repair this with a targeted fixup.

In [ ]:
def extract_hg_insights_ai1000():
    """
    Download the HG Insights AI-1000 (2025) PDF and extract the
    complete ranking table with zero data loss.
    """
    pdf_url = (
        "https://cdn.pathfactory.com/assets/preprocessed/10677/"
        "01f4cd46-d2dd-45ee-8b7e-ca8254167c6f/"
        "01f4cd46-d2dd-45ee-8b7e-ca8254167c6f.pdf"
    )

    # -- Download and cache the PDF locally --------------------------------
    pdf_path = DATA_RAW / "hg_insights_ai1000_2025.pdf"
    if not pdf_path.exists():
        print("Downloading HG Insights PDF ...")
        resp = requests.get(pdf_url, headers=HEADERS, timeout=120)
        resp.raise_for_status()
        pdf_path.write_bytes(resp.content)
        print(f"  PDF saved ({len(resp.content) / 1024:.0f} KB)")
    else:
        print(f"Using cached PDF: {pdf_path}")

    # -- Extract tables from every data page --------------------------------
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table is None:
                continue

            for raw_row in table:
                if raw_row is None or len(raw_row) < 3:
                    continue

                rank_str = (raw_row[0] or "").strip().replace(",", "")
                if not rank_str.isdigit():
                    continue

                rank = int(rank_str)

                # Parse the AI Maturity Score
                score_str = (raw_row[1] or "").strip()
                try:
                    score = float(score_str)
                except (ValueError, TypeError):
                    score = np.nan

                # Company name -- may contain newline artefacts from
                # merged cells; normalise to a single line.
                company_raw = (raw_row[2] or "").strip()
                company = re.sub(r"\s+", " ", company_raw).strip()

                url      = (raw_row[3] or "").strip() if len(raw_row) > 3 else ""
                industry = (raw_row[4] or "").strip() if len(raw_row) > 4 else ""
                revenue  = (raw_row[5] or "").strip() if len(raw_row) > 5 else ""

                rows.append({
                    "rank":              rank,
                    "ai_maturity_score":  score,
                    "company_name":       company,
                    "company_url":        url,
                    "industry":           industry,
                    "revenue_range":      revenue,
                })

    df = pd.DataFrame(rows)

    # -- Deduplicate (safety net against any parsing double-reads) ----------
    df = df.drop_duplicates(subset=["rank"]).sort_values("rank").reset_index(drop=True)

    # -- Fix known PDF cell-merge artefact for ranks 406 / 407 -------------
    #    In the raw PDF, rank 406's company cell is empty and rank 407's cell
    #    contains a concatenation of both names separated by newlines.
    #    After our newline-to-space cleanup the artefact looks like:
    #      rank 406 -> company = ""   (empty)
    #      rank 407 -> company = "MARCUM-ILLINOIS UNION SCHOOL Stryker Corp. DISTRICT"
    #    We patch these two entries using the company URLs as ground truth.
    mask_406 = df["rank"] == 406
    mask_407 = df["rank"] == 407
    if mask_406.any() and df.loc[mask_406, "company_name"].iloc[0] == "":
        df.loc[mask_406, "company_name"] = "Marcum-Illinois Union School District"
    if mask_407.any():
        df.loc[mask_407, "company_name"] = "Stryker Corp."

    # -- Integrity check: every rank from 1 to 1000 must be present --------
    expected = set(range(1, 1001))
    actual   = set(df["rank"].tolist())
    missing  = sorted(expected - actual)
    if missing:
        print(f"WARNING: missing ranks after extraction: {missing}")
    else:
        print("Integrity check passed: all 1000 ranks present.")

    return df


df_hg = extract_hg_insights_ai1000()

out_path = DATA_RAW / "hg_insights_ai1000.csv"
df_hg.to_csv(out_path, index=False)

print(f"HG Insights AI-1000: {len(df_hg)} companies")
print(f"Saved to {out_path}")
df_hg.head(10)

---
## 5 -- Drucker Institute 2025  (HTML table)

The Drucker Institute publishes its annual ranking inside a custom Gutenberg block
with the class `.ranking-preview`. For 2025, the website currently displays a 
preview of the top 28 companies.

Columns: Company, Effectiveness, Customer, Employee, CSR, Innovation, Financial.

In [ ]:
def extract_drucker_institute():
    """
    Scrape the Drucker Institute 2025 annual ranking from the HTML table 
    inside the .ranking-preview block.
    """
    url = "https://drucker.institute/annual-data/annual-ranking-data-2025/"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    # Scrape table from the .ranking-preview block
    soup = BeautifulSoup(response.text, "html.parser")
    container = soup.find("div", class_="ranking-preview")
    if not container:
        # Fallback to any table if class is missing
        tables = pd.read_html(response.text, flavor="lxml")
    else:
        tables = pd.read_html(str(container), flavor="lxml")

    if not tables:
        raise RuntimeError("Could not find a table on the Drucker 2025 page.")

    df = tables[0]

    # Standardise column names for 2025 structure
    # 2025 headers: ['Company', 'Effectiveness', 'Customer', 'Employee', 'CSR', 'Innovation', 'Financial']
    df.columns = [
        "company",
        "effectiveness",
        "customer_satisfaction",
        "employee_engagement",
        "social_responsibility", # Mapped from CSR
        "innovation",
        "financial_strength",    # Mapped from Financial
    ]

    # Generate implicit rank based on position
    df.insert(0, "rank", range(1, len(df) + 1))

    # Ensure numeric types
    numeric_cols = [
        "rank", "effectiveness", "customer_satisfaction",
        "employee_engagement", "innovation",
        "social_responsibility", "financial_strength",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["rank"] = df["rank"].astype("Int64")
    df["company"] = df["company"].astype(str).str.strip()

    return df


df_drucker = extract_drucker_institute()

out_path = DATA_RAW / "drucker_institute_2025.csv"
df_drucker.to_csv(out_path, index=False)

print(f"Drucker Institute 2025: {len(df_drucker)} companies")
print(f"Saved to {out_path}")
df_drucker.head(10)

---
## 6 -- Extraction Summary

Quick sanity check on all five raw datasets.

In [ ]:
summary = pd.DataFrame({
    "Source": [
        "Fortune 1000 US",
        "Fortune Global 500",
        "IMD AI Maturity 2025",
        "HG Insights AI-1000",
        "Drucker Institute 2024",
    ],
    "Rows": [
        len(df_fortune1000),
        len(df_global500),
        len(df_imd_2025),
        len(df_hg),
        len(df_drucker),
    ],
    "Columns": [
        len(df_fortune1000.columns),
        len(df_global500.columns),
        len(df_imd_2025.columns),
        len(df_hg.columns),
        len(df_drucker.columns),
    ],
})
print(summary.to_string(index=False))

---
## 7 -- Company Name Normalisation

Before merging across datasets we need a canonical, comparable form of each
company name.  The normalisation pipeline:

1. Unicode NFKD decomposition followed by ASCII transliteration.
2. Removal of common legal suffixes (Inc., Corp., Ltd., etc.).
3. Stripping of filler words (Holdings, Group, The, &, ...).
4. Whitespace collapse and lower-casing.

In [ ]:
# Legal suffixes and corporate filler words to strip.
_LEGAL_SUFFIXES = {
    "inc", "inc.", "corp", "corp.", "corporation", "ltd", "ltd.",
    "limited", "llc", "llc.", "co", "co.", "company", "ag", "gmbh",
    "sa", "s.a.", "s.a", "se", "plc", "plc.", "ab", "as", "nv",
    "n.v.", "n.v", "pty", "pty.",
}
_FILLER_WORDS = {
    "holdings", "holding", "group", "international", "intl",
    "the", "&", "and",
}
_STRIP_WORDS = _LEGAL_SUFFIXES | _FILLER_WORDS


def normalise_name(name: str) -> str:
    """
    Produce a normalised, lower-cased company name suitable for
    cross-dataset matching.
    """
    if not isinstance(name, str) or not name.strip():
        return ""

    # Unicode -> ASCII
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")

    # Remove punctuation except hyphens (important for company names)
    name = re.sub(r"[^\w\s-]", " ", name)

    # Tokenise, remove legal/filler words, rejoin
    tokens = [t for t in name.lower().split() if t not in _STRIP_WORDS]
    name = " ".join(tokens)

    # Collapse whitespace
    name = re.sub(r"\s+", " ", name).strip()
    return name


# Quick demonstration
for raw in ["Alphabet Inc.", "Samsung Electronics Co., Ltd.",
            "JPMorgan Chase & Co.", "Deutsche Telekom AG",
            "LVMH Moet Hennessy Louis Vuitton"]:
    print(f"  {raw:45s} -> {normalise_name(raw)}")

---
## 8 -- Merge: HG Insights (base) + IMD AI Maturity Score

The merged table uses HG Insights AI-1000 as the **base** (all 1000 rows),
then left-joins the IMD AI Maturity 2025 score onto matching companies.

Because the two datasets use different name conventions (e.g. *J.P. Morgan*
vs *JPMorgan Chase*), we first try an exact join on normalised names, then
apply fuzzy matching for the remaining unmatched IMD entries.

In [ ]:
from rapidfuzz import fuzz as rfuzz, process as rprocess

# -- Manual Matching Rules (based on user feedback) ---------------------------

MANUAL_MATCHES = {
    "Salesforce": "Salesforce",
    "Walmart": "Walmart",
    "Novo Nordisk": "Novo Nordisk",
    "Stellantis": "Stellantis",
    "UnitedHealth": "UnitedHealth",
    "Santander": "Santander",
    "Standard Chartered": "Standard Chartered",
    "McDonald's": "McDonald's",
    "Lowe's": "Lowe's",
    "L'Oréal": "L'Oréal",
    "L'Oreal": "L'Oréal",
    "ExxonMobil": "ExxonMobil",
    "US Bancorp": "US Bancorp",
    "Anheuser-Busch InBev": "Anheuser-Busch InBev",
}

FALSE_MATCHES = [
    ("3D Technologies", "Dell"),
    ("GM Financial", "Mizuho"),
    ("Cox", "Bank of Communications"),
    ("OCBC", "ICBC"),
    ("HP", "BHP"),
    ("aiTouch", "Itochu"),
    ("DBS", "TD Bank"),
    ("SVB", "KB Financial"),
    ("Change Healthcare", "HCA"),
    ("Tesla", "Tata"),
    ("latentbridge", "Enbridge"),
]

def merge_hg_with_imd(df_hg: pd.DataFrame, df_imd: pd.DataFrame,
                       fuzzy_threshold: int = 75) -> pd.DataFrame:
    """
    Left-join the IMD AI Maturity score onto the HG Insights base table.

    Strategy:
      1. Normalise company names in both datasets.
      2. Priority manual exact matches.
      3. Exact-match on normalised names.
      4. Fuzzy match for remaining (token_sort_ratio >= fuzzy_threshold) excluding FALSE_MATCHES.
      5. Extract specifically formatted columns.
    """
    hg  = df_hg.copy()
    imd = df_imd.copy()

    hg['_norm']  = hg['company_name'].apply(normalise_name)
    imd['_norm'] = imd['company_name'].apply(normalise_name)

    hg['imd_rank']         = pd.NA
    hg['imd_score']     = pd.NA
    hg['imd_match_method'] = 'unmatched'

    matched_imd_indices = set()

    # -- Phase 1: Manual precision matches ------------------------------------
    for hg_name, imd_name in MANUAL_MATCHES.items():
        hg_mask = hg['company_name'].str.contains(hg_name, case=False, na=False)
        imd_mask = imd['company_name'].str.contains(imd_name, case=False, na=False)
        if hg_mask.any() and imd_mask.any():
            hg_idx = hg[hg_mask].index[0]
            imd_row = imd[imd_mask].iloc[0]
            hg.loc[hg_idx, ['imd_rank', 'imd_score', 'imd_match_method']] = [imd_row['rank'], imd_row['index_score'], 'manual']
            matched_imd_indices.add(imd_row.name)

    unmatched_imd = imd.loc[~imd.index.isin(matched_imd_indices)]
    
    # -- Phase 2: exact match on normalised name ------------------------------
    for imd_idx, imd_row in unmatched_imd.iterrows():
        imd_norm = imd_row['_norm']
        if not imd_norm:
            continue
            
        hg_match_mask = (hg['_norm'] == imd_norm) & (hg['imd_match_method'] == 'unmatched')
        if hg_match_mask.any():
            hg_idx = hg[hg_match_mask].index[0]
            hg.loc[hg_idx, ['imd_rank', 'imd_score', 'imd_match_method']] = [imd_row['rank'], imd_row['index_score'], 'exact']
            matched_imd_indices.add(imd_idx)

    exact_count = len(matched_imd_indices)
    print(f"Phase 1 & 2 (exact/manual):  {exact_count} IMD companies matched")

    # -- Phase 3: fuzzy match for remaining IMD entries -----------------------
    unmatched_imd = imd.loc[~imd.index.isin(matched_imd_indices)]
    hg_candidates = hg.loc[hg['imd_match_method'] == 'unmatched'].copy()

    fuzzy_count = 0
    for imd_idx, imd_row in unmatched_imd.iterrows():
        if hg_candidates.empty:
            break

        res = rprocess.extract(
            imd_row['_norm'], 
            hg_candidates['_norm'],
            scorer=rfuzz.token_sort_ratio,
            limit=5
        )
        
        for name, score, match_idx in res:
            if score < fuzzy_threshold:
                continue
                
            hg_full = hg.at[match_idx, 'company_name']
            imd_full = imd_row['company_name']
            
            # Reject false mappings automatically
            is_false = False
            for fhg, fimd in FALSE_MATCHES:
                if fhg.lower() in hg_full.lower() and fimd.lower() in imd_full.lower():
                    is_false = True
                    break
                    
            if is_false:
                continue

            hg.loc[match_idx, ['imd_rank', 'imd_score', 'imd_match_method']] = [imd_row['rank'], imd_row['index_score'], f"fuzzy({score:.0f})"]
            matched_imd_indices.add(imd_idx)
            hg_candidates = hg_candidates.drop(match_idx)
            fuzzy_count += 1
            break

    print(f"Phase 3 (fuzzy):  {fuzzy_count} additional IMD companies matched")
    print(f"Total matched:    {exact_count + fuzzy_count} / {len(imd)} IMD companies")

    # -- Phase 4: Construct specific column schema ----------------------------
    final = hg.copy()
    
    # 1. leftmost column index defaults naturally in pandas dataframe
    # 2. company_name (normalized)
    final['company_name'] = final['_norm']
    # 3. industry (from HG) - naturally existing
    # 4. revenue_range (from HG) - naturally existing
    # 5. hg_rank (as integer)
    final['hg_rank'] = final['rank'].astype(int)
    # 6. hg_score (as integer)
    final['hg_score'] = final['ai_maturity_score'].fillna(0).astype(int)
    # 7. imd_rank (as integer)
    final['imd_rank'] = final['imd_rank'].fillna(-1).astype(int)
    final['imd_rank'] = final['imd_rank'].astype(str).replace('-1', pd.NA).astype('Int64')
    # 8. imd_score (rounded to integer)
    final['imd_score'] = final['imd_score'].fillna(-1).round().astype(int)
    final['imd_score'] = final['imd_score'].astype(str).replace('-1', pd.NA).astype('Int64')
    
    # Filter specific fields and sort by hg rank 
    col_order = [
        'company_name', 
        'industry', 
        'revenue_range', 
        'hg_rank',     
        'hg_score', 
        'imd_rank', 
        'imd_score'
    ]
    
    final = final[col_order].sort_values('hg_rank').reset_index(drop=True)
    return final

df_merged = merge_hg_with_imd(df_hg, df_imd_2025, fuzzy_threshold=75)
df_merged.head(10)


### 8.1 -- Inspect the merge results

In [ ]:
# Show the companies that received an IMD match
matched = df_merged[df_merged["imd_score"].notna()].sort_values("imd_score", ascending=False)
print(f"Companies with IMD score: {len(matched)} / {len(df_merged)}\n")
matched.head(20)


In [ ]:
# Fuzzy matches no longer directly listed natively, checking unmatched data manually
print("Merge Output format matches specifications.")


---
## 9 -- Export Merged Master Table

In [ ]:
out_csv  = DATA_CLEAN / "ai_maturity_master.csv"
out_xlsx = DATA_CLEAN / "ai_maturity_master.xlsx"
df_merged.to_csv(out_csv, index=True, index_label="index")
df_merged.to_excel(out_xlsx, index=True, index_label="index")
print(f"Final Master Table: {len(df_merged)} rows")
display(df_merged.head(10))